# EWC experiments on Colab GPU

Runs one condition from `EWC_EXPERIMENT_CONDITIONS` (set `CONDITION` and `SEEDS` in cell 5), ANN and SNN.
Code is cloned from GitHub, so **commit and push local changes first**.

In [5]:
# 1. Confirm this kernel is the Colab T4, not your laptop.
import platform, torch
print("machine:", platform.node(), platform.system())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: select the Colab kernel and a T4 GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

machine: 691402142ae6 Linux
torch: 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4


In [6]:
# 2. Get the latest code from GitHub and install the one missing package.
import os, sys
REPO = "/content/Dissertation"
if not os.path.exists(REPO):
    !git clone -q https://github.com/ghonilcloud/Dissertation.git {REPO}
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q spikingjelly
sys.path.insert(0, REPO)

/content/Dissertation
9f958a8 (HEAD -> main, origin/main, origin/HEAD) got ewc working, added replay, did more experiments


In [11]:
# 3. Stop early if GitHub still has the old code (conditions without ewc_lambda/replay).
import ewc.runEwcExperiments as r
assert "replay" in r.EWC_EXPERIMENT_CONDITIONS[0], "Old code on GitHub: commit and push, then re-run cell 2."
print(r.EWC_EXPERIMENT_CONDITIONS[0])

{'epochs': 1, 'time_steps': 2, 'ewc_lambda': 100.0, 'replay': False}


In [9]:
# 4. Download CIFAR-10 (data/ is not in git).
# Fast path: the archive attached to the repo's GitHub release (tag "cifar10").
# torchvision then finds it, checks its MD5, skips its own slow download and just extracts.
# If the release is missing, torchvision falls back to the (slow) Toronto server.
!mkdir -p data
!rm -f data/cifar-10-python.tar.gz  # remove any half-finished download
!wget -q --show-progress -O data/cifar-10-python.tar.gz https://github.com/ghonilcloud/Dissertation/releases/download/cifar10/cifar-10-python.tar.gz || rm -f data/cifar-10-python.tar.gz
from utilities.dataset import get_cifar_datasets
_ = get_cifar_datasets("cifar10", "data", download=True)

data/cifar-10-pytho 100%[===================>] 162.60M  26.6MB/s    in 6.8s    


In [ ]:
# 5. Run condition 3 (5 epochs, 7 SNN time steps), ANN + SNN, task scenario.
import importlib, time
importlib.reload(r)  # undo any condition/seed changes from earlier runs of this cell
CONDITION = 3        # 1-based, matches EWC_EXPERIMENT_CONDITIONS in runEwcExperiments.py
SEEDS = (42, 123, 456, 789, 1011)  # all 5 seeds (same as the baselines)
OUTPUT = f"results/colab_condition{CONDITION}.json"

r.EWC_EXPERIMENT_CONDITIONS = r.EWC_EXPERIMENT_CONDITIONS[CONDITION - 1:CONDITION]
r.DEFAULT_SEEDS = SEEDS
print(r.EWC_EXPERIMENT_CONDITIONS[0], "seeds", SEEDS)
start = time.perf_counter()
res = r.run_grid(output_path=OUTPUT, progress=True)
wall = time.perf_counter() - start
c = res["conditions"][0]
for m in ("ann", "snn"):
    print(f"{m}: final accuracy {c[m]['mean_final_accuracy']:.3f}, training time {c[m]['training_time_total_seconds']:.1f}s")
print(f"wall time (incl. EWC Fisher + evaluation): {wall:.1f}s")

In [ ]:
# 6. Save the result to your computer (Colab files are deleted when the session ends).
from google.colab import files
files.download(OUTPUT)